In [100]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

library(mixOmics)
library(BiocParallel)
library(svglite)

In [101]:
DATA_DIR <- "/home/mariya/Data/pallas"
MIXOMICS_DIR <- file.path(DATA_DIR, "mixOmics/models_20260813")
PREDICTIONS_OUTPUT_DIR <- "/home/mariya/Development/pallas/reports/output/predictions/bimodal"
REPORT_OUTPUT_DIR <- "/home/mariya/Development/pallas/reports/output/evaluation/bimodal"

DATA_DIR <- file.path(DATA_DIR, "mlomix_datasets")

UNLABELED_DATA_DIR <- file.path(DATA_DIR, "unlabeled-bimodal-reduced")
UNLABELED_METH_FILE <- file.path(UNLABELED_DATA_DIR, "features.dnam.csv")
UNLABELED_GEX_FILE <- file.path(UNLABELED_DATA_DIR, "features.gex.csv")

GEX_MODEL_FILE <- file.path(MIXOMICS_DIR, "gex_model.rds")
DNAM_MODEL_FILE <- file.path(MIXOMICS_DIR, "dnam_model.rds")
DIABLO_MODEL_FILE <- file.path(MIXOMICS_DIR, "diablo_model.rds")

DIST <- "centroids.dist"

# Make output directory if it doesn't exist
if (!dir.exists(PREDICTIONS_OUTPUT_DIR)) {
    dir.create(PREDICTIONS_OUTPUT_DIR, recursive = TRUE)
}

In [102]:
gex_model    <- readRDS(GEX_MODEL_FILE)
dnam_model   <- readRDS(DNAM_MODEL_FILE)
diablo_model <- readRDS(DIABLO_MODEL_FILE)


In [103]:
source("helpers.R")
test_data <- unlabeled_data(UNLABELED_GEX_FILE, UNLABELED_METH_FILE)

GEX features: 1121 
DNAm features: 1186 


In [104]:
X_gex_test <- test_data$X_gex
X_dnam_test <- test_data$X_dnam

head(X_gex_test)
head(X_dnam_test)

,ENSG00000184507,ENSG00000079102,ENSG00000248329,ENSG00000101115,ENSG00000172752,ENSG00000097007,ENSG00000132185,ENSG00000167880,ENSG00000177469,ENSG00000188064,⋯,ENSG00000170396,ENSG00000130702,ENSG00000187325,ENSG00000169213,ENSG00000186980,ENSG00000182747,ENSG00000144681,ENSG00000072954,ENSG00000181007,ENSG00000107819
ALL_1001,-1.528217,-1.772906,-1.772906,-1.77290585,-1.556421,5.744491,1.1535998,-0.9028575,0.07788697,-1.772906,⋯,-1.5447673,-0.9575067,5.705442,-1.689974,-1.772906,-0.3641423,-1.3420567,2.4291750,3.453165,4.152621
ALL_1006,-1.772906,-1.772906,-1.772906,-0.65733323,-1.508822,6.309054,-0.2359833,2.1937585,-1.15082343,-1.642353,⋯,-1.7729058,-0.5422928,6.270758,-1.321829,-1.772906,-1.5034500,-1.7729058,-0.9297546,3.067561,3.392129
ALL_1034,-1.772906,-1.711405,-1.491250,4.09825741,-1.664681,5.808953,1.1920234,-1.3958297,-1.07051306,-1.772906,⋯,-0.9055206,-0.4728739,5.757177,-1.693012,-1.772906,-1.7729058,-1.7729058,0.9094844,2.476011,3.405101
ALL_1089,-1.772906,-1.680180,-1.772906,-0.15503761,-1.465495,6.298213,-0.3242633,0.5333668,0.54423807,-1.619781,⋯,-1.3114167,1.0283792,4.070370,-1.342975,-1.772906,-1.2018968,-0.1085736,-0.1569802,1.900471,1.299506
ALL_219,-1.772906,-1.772906,-1.772906,-1.49261353,-1.053895,3.922017,0.3605152,-0.6102910,-0.32197282,-1.772906,⋯,-0.9086588,0.2539844,2.824904,-1.642708,-1.772906,-1.7729058,-1.7729058,-1.2786745,0.984353,1.294686
ALL_232,-1.396033,-1.108682,-1.327374,0.07658081,-1.295523,5.640666,-1.7729058,2.2002541,-0.31119389,-1.772906,⋯,-1.1374826,1.0621085,3.607490,-1.305896,-1.772906,-1.7729058,-1.1294331,0.9899996,2.623760,3.715946


,cg10443049,cg07275648,cg00760938,cg04646674,cg12183594,cg03732020,cg14992144,cg13799263,cg08166720,cg00015639,⋯,cg01955153,cg26092233,cg18631798,cg21927946,cg11551919,cg01883662,cg20294319,cg12737392,cg09291834,cg11414821
ALL_1001,0.7609,0.7150,0.2617,0.4517,0.9839,0.9421,0.0821,0.7733,0.9370,0.8741,⋯,0.1940,0.2133,0.1246,0.3154,0.6706,0.9473,0.0783,0.8018,0.4402,0.0366
ALL_1006,0.9026,0.1110,0.0407,0.5353,0.9853,0.7225,0.0208,0.9209,0.6786,0.7467,⋯,0.1687,0.0597,0.0345,0.0536,0.9831,0.8112,0.1079,0.2616,0.0965,0.0338
ALL_1034,0.7081,0.7621,0.0524,0.9395,0.9832,0.5669,0.0242,0.9136,0.9502,0.9173,⋯,0.3504,0.0533,0.2578,0.8914,0.9690,0.9632,0.0672,0.0968,0.0562,0.0325
ALL_1089,0.8813,0.3624,0.2346,0.9056,0.9844,0.9288,0.0409,0.5518,0.9354,0.8817,⋯,0.1820,0.2131,0.1129,0.3556,0.3615,0.5059,0.0725,0.1410,0.0210,0.0350
ALL_219,0.3769,0.9403,0.4972,0.6915,0.9783,0.7390,0.1798,0.8788,0.9501,0.8669,⋯,0.2307,0.5644,0.5097,0.0469,0.9382,0.8446,0.0513,0.2497,0.0194,0.0561
ALL_232,0.8850,0.8288,0.2439,0.6653,0.9842,0.7086,0.1182,0.9148,0.9612,0.5844,⋯,0.1906,0.1563,0.2035,0.0492,0.6731,0.8919,0.0542,0.4563,0.2122,0.0454


In [105]:
# X_gex_test  <- X_gex_test[,  colnames(gex_model$X)]
# X_dnam_test <- X_dnam_test[, colnames(dnam_model$X)]

# Verify
cat("GEX names match: ",  all(colnames(X_gex_test)  == colnames(gex_model$X)),  "\n")
cat("DNAm names match:",  all(colnames(X_dnam_test) == colnames(dnam_model$X)), "\n")

# Show number of features in test set vs model
cat("GEX features in test set: ", ncol(X_gex_test), " vs model: ", ncol(gex_model$X), "\n")
cat("DNAm features in test set: ", ncol(X_dnam_test), " vs model: ", ncol(dnam_model$X), "\n")

GEX names match:  TRUE 
DNAm names match: TRUE 
GEX features in test set:  1121  vs model:  1121 
DNAm features in test set:  1186  vs model:  1186 


In [106]:

gex_pred    <- predict(gex_model,    X_gex_test,  dist = DIST)
dnam_pred   <- predict(dnam_model,   X_dnam_test, dist = DIST)
diablo_pred <- predict(diablo_model, newdata = list(gex = X_gex_test,
                                                     dnam = X_dnam_test),
                                                     dist = DIST)

In [107]:
# Extract predicted classes at final component
ncomp_gex_final    <- gex_model$ncomp
ncomp_dnam_final   <- dnam_model$ncomp
ncomp_diablo_final <- diablo_model$ncomp

gex_class    <- gex_pred$class$centroids.dist[,    ncomp_gex_final]
dnam_class   <- dnam_pred$class$centroids.dist[,   ncomp_dnam_final]
diablo_class <- diablo_pred$WeightedVote$centroids.dist[, ncol(diablo_pred$WeightedVote$centroids.dist)]

In [108]:
# Make predictiosn dir if not exist
if (!dir.exists(PREDICTIONS_OUTPUT_DIR)) {
    dir.create(PREDICTIONS_OUTPUT_DIR, recursive = TRUE)
}

# After true_label, create three cols: gex_pred, dnam_pred, diablo_pred
# that contain the prediction from the MAX_COMP component for each model
# Keep the rest of the columns as well
predictions_df <- data.frame(
    "Sample" = rownames(X_gex_test),
    "GEX sPLS-DA prediction" = gex_class,
    "DNAm sPLS-DA prediction" = dnam_class,
    "DIABLO sPLS-DA prediction" = diablo_class,
    "GEX" = gex_pred$class,
    "DNAm" = dnam_pred$class,
    "DIABLO" = diablo_pred$class,
    check.names = FALSE
)

predictions_df <- predictions_df[order(predictions_df$Sample), ]

write.csv(predictions_df, file.path(PREDICTIONS_OUTPUT_DIR, "bimodal_predictions_unlabeled.csv"), row.names = FALSE)

In [109]:
# Build results data frame
results_df <- data.frame(
  sample_id   = rownames(X_gex_test),
  gex_pred    = as.character(gex_class),
  dnam_pred   = as.character(dnam_class),
  diablo_pred = as.character(diablo_class),
  row.names   = NULL
)

head(results_df)

,sample_id,gex_pred,dnam_pred,diablo_pred
,<chr>,<chr>,<chr>,<chr>
1,ALL_1001,"B-ALL, PAX5alt","B-ALL, BCR::ABL1","B-ALL, PAX5alt"
2,ALL_1006,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid"
3,ALL_1034,"B-ALL, iAMP21","B-ALL, BCR::ABL1","B-ALL, iAMP21"
4,ALL_1089,"B-ALL, PAX5alt","B-ALL, PAX5alt","B-ALL, PAX5alt"
5,ALL_219,"AML, KMT2A-r","B-ALL, BCR::ABL1","AML, NUP98-r"
6,ALL_232,"B-ALL, BCR::ABL1-like","B-ALL, BCR::ABL1","B-ALL, BCR::ABL1-like"
